# Build an Agentic Corrective RAG System with LangGraph

| Property | Value |
|---|---|
| Origin | Yan et al., *Corrective Retrieval Augmented Generation* (2024). [arXiv:2401.15884](https://arxiv.org/abs/2401.15884) |

This project will cover a full hands-on workflow and demonstration of how to build an Agentic Corrective RAG (CRAG) System with LangGraph

The idea would be to implement the workflow taking inspiration from the [Corrective Retrieval Augmented Generation](https://arxiv.org/pdf/2401.15884) research paper.

The main challenge of RAG systems include:

- Poor Retrieval can lead to issues in LLM response generation
- Bad retrieval or lack of information in the vector database can also lead to out of context or hallucinated answers

The idea is to couple a RAG system with a few checks in place and perform web searches if there is a lack of relevant context documents to the given user query.

We can build this as an agentic RAG system by having a specific functionality step as a node in the graph and use LangGraph to implement it. Key steps in the node will include prompts being sent to LLMs to perform specific tasks as seen in the detailed workflow below:

![](https://i.imgur.com/oAfXksw.png)


### Agentic Corrective RAG System Workflow

This project implements an **Agentic Corrective RAG System** that enhances the reliability and precision of responses by combining document grading, aumenting web search, and RAG. The system ensures only high-quality, grounded answers are generated even when the initial context retrieved from the vector database is incomplete or irrelevant.

The workflow includes the following components:

1. **Document Retrieval and Grading**:
   - A user query is first sent to a **Vector Database** to retrieve relevant documents.
   - These documents are passed through an **LLM Grader Prompt**:
     - The LLM evaluates each document and labels it as either **'yes'** (relevant) or **'no'** (irrelevant) based on its usefulness for answering the query.
     - This filtering step ensures that only the most relevant documents are retained for response generation.

2. **Dynamic Decision Routing**:
   - A **Decision Node** checks the document grading results:
     - If **> 50% of retrieved documents are relevant**, the system proceeds with a **standard RAG Prompt**, using only those documents to generate an answer.
     - If **<= 50% of retrieved documents are relevant**, the system switches to a **fallback corrective workflow**.

3. **Query Rephrasing and Web Search**:
   - When retrieved context is inadequate:
     - The original query is sent to the LLM with a **Rephrase Prompt** to generate a more search-optimized version of the query.
     - The rephrased query is passed to a **Web Search Tool** to retrieve fresh and more relevant context documents from the web.
     - These web-retrieved documents are then combined with any relevant retrieved context documents from the Vector DB and used as the final context documents

4. **Final Answer Generation**:
   - Whether using documents from the vector database or web search, the final step involves sending the query and relevant context documents into the **RAG Prompt**, which instructs the LLM to:
     - Use only the given documents to answer the question.
     - Avoid making up information or hallucinating unsupported content.

This agentic workflow adds an additional layer of control and recovery to the RAG pipeline, ensuring more accurate responses in dynamic and unpredictable retrieval scenarios.



___Created By: [Dipanjan (DJ)](https://www.linkedin.com/in/dipanjans/)___


## Install OpenAI, Tavily, LangGraph and LangChain dependencies


In [ ]:
!pip install langchain==0.3.20
!pip install langchain-openai==0.3.9
!pip install langchain-community==0.3.20
!pip install langgraph==0.3.18
!pip install langchain-tavily==0.1.5

## Install PyMuPDF for loading PDF documents

In [ ]:
!pip install pymupdf==1.25.4

## Install ChromaDB LangChain Wrapper for Vector DB

In [ ]:
!pip install langchain-chroma==0.2.2

## Enter Open AI API Key

In [ ]:
from getpass import getpass

OPENAI_KEY = getpass('Enter Open AI API Key: ')

## Enter Tavily Search API Key

Get a free API key from [here](https://tavily.com/#api)

In [ ]:
TAVILY_API_KEY = getpass('Enter Tavily Search API Key: ')

## Setup Environment Variables

In [ ]:
import os

os.environ['OPENAI_API_KEY'] = OPENAI_KEY
os.environ['TAVILY_API_KEY'] = TAVILY_API_KEY

## Build a Search Index for Research Paper Data

We will build a vector database for retrieval and search by indexing a few research paper documents, similar to any standard RAG workflows

### Open AI Embedding Models

LangChain enables us to access Open AI embedding models which include the newest models: a smaller and highly efficient `text-embedding-3-small` model, and a larger and more powerful `text-embedding-3-large` model.

In [ ]:
from langchain_openai import OpenAIEmbeddings

# details here: https://openai.com/blog/new-embedding-models-and-api-updates
openai_embed_model = OpenAIEmbeddings(model='text-embedding-3-small')

### Get the research paper data

In [ ]:
# if you can't download using the following code
# go to https://drive.google.com/file/d/1ZOtPmuR-2KpzPvkiQiTVxAyJFo6NszG-/view?usp=sharing download it
# manually upload it on colab

!gdown 1ZOtPmuR-2KpzPvkiQiTVxAyJFo6NszG-

In [ ]:
!unzip research_papers.zip

### Load and Chunk Documents

We create a directory loader to use a PDF loader (using pymupdf) and load all PDF documents from a given folder

In [ ]:
from langchain_community.document_loaders import DirectoryLoader

# Define a function to create a DirectoryLoader for a specific file type
def create_directory_loader(file_type, directory_path, loader_class, loader_args):
    return DirectoryLoader(
        path=directory_path,
        glob=f"**/*{file_type}",
        loader_cls=loader_class,
        loader_kwargs=loader_args,
        show_progress=True
    )

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

pdf_extn = '.pdf'
pdf_loader_class = PyMuPDFLoader
pdf_loader_args = {} # in case you want to change any settings in pymupdfloader
directory= './research_papers'

pdf_loader = create_directory_loader(file_type=pdf_extn,
                                     directory_path=directory,
                                     loader_class=pdf_loader_class,
                                     loader_args=pdf_loader_args)

# load docs
docs = pdf_loader.load()
len(docs) # PyMuPDF loads every document and breaks it per page by default

In [ ]:
docs[0]

We then use standard recursive character text chunking

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Chunk docs
splitter = RecursiveCharacterTextSplitter(chunk_size=4000, chunk_overlap=300)
chunked_docs = splitter.split_documents(docs)

In [ ]:
len(chunked_docs)

In [ ]:
chunked_docs[:3]

### Create a Vector DB and persist on disk

Here we initialize a connection to a Chroma vector DB client, and also we want to save to disk, so we simply initialize the Chroma client and pass the directory where we want the data to be saved to.

In [ ]:
from langchain_chroma import Chroma

# create vector DB of docs and embeddings - takes < 30s on Colab
chroma_db = Chroma.from_documents(documents=chunked_docs,
                                  collection_name='rag_db',
                                  embedding=openai_embed_model,
                                  # need to set the distance function to cosine else it uses euclidean by default
                                  # check https://docs.trychroma.com/guides#changing-the-distance-function
                                  collection_metadata={"hnsw:space": "cosine"},
                                  persist_directory="./rag_db")

### Setup a Vector Database Retriever

Here we use the following retrieval strategy:

- Similarity with Threshold Retrieval


### Similarity with Threshold Retrieval

We use cosine similarity here and retrieve the top 5 similar documents based on the user input query and also introduce a cutoff to not return any documents which are below a certain similarity threshold

In [ ]:
similarity_threshold_retriever = chroma_db.as_retriever(search_type="similarity_score_threshold",
                                                        search_kwargs={"k": 5,
                                                                       "score_threshold": 0.35})

### Test out a few queries

In [ ]:
query = "what is PEFT?"
topk_docs = similarity_threshold_retriever.invoke(query)
topk_docs

Looks like it is getting relevant documents from the vector DB

In [ ]:
query = "what is time series forecasting?"
topk_docs = similarity_threshold_retriever.invoke(query)
topk_docs

Whoops seems like there exists no relevant docs for this. The Agentic Corrective RAG should still be able to handle queries like this

In [ ]:
query = "what are popular patterns for Agentic AI?"
topk_docs = similarity_threshold_retriever.invoke(query)
topk_docs

None of the above documents are relevant to the query. The Agentic Corrective RAG System should be able to handle this also

## Create AI Workflows and Tools for Key Components in our Agentic RAG System

There are a few AI workflows (sequential pipelines) and tools we would need to create which we will be using in different steps in our Agentic Corrective RAG workflow. These include:

- **Query Retrieval Grader Workflow:** This is an essential workflow which can take in a user query, a list of retrieved documents from the vector DB and grade each query as 'yes' or 'no' based on if the document is relevant to the user query or not

- **QA RAG Workflow:** This is a workflow which can take in a user query, list of retrived context documents and use a standard RAG workflow where an LLM uses these context documents to generate a contextual response for the user query

- **Query Rephraser Workflow:** This is a workflow which uses an LLM to rephrase the given user query (if needed) and make it more optimized for web search

- **Web Search Tool:** Build a custom tool which uses the Tavily Search API to search for a user query, get the top web page results and also extract the text content from those web pages

Most of these workflows will be built as LangChain chains (pipelines)

### Create a Query Retrieval Grader Workflow

Here we will use an LLM itself to grade if any retrieved document is relevant to the given user query - Answer will be either `yes` or `no` for each document

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI


# Data model for LLM output format
class GradeDocuments(BaseModel):
    """Binary score for relevance check on retrieved documents."""
    binary_score: str = Field(
        description="Documents are relevant to the question, 'yes' or 'no'"
    )


# LLM for grading
llm = ChatOpenAI(model="gpt-4o", temperature=0)
structured_llm_grader = llm.with_structured_output(GradeDocuments)

# Prompt template for grading
SYS_PROMPT = """You are an expert grader assessing relevance of a retrieved document to a user question.
                Follow these instructions for grading:
                  - If the document contains keyword(s) or semantic meaning related to the question, grade it as relevant.
                  - The overall grade should focus more on the semantic meaning rather than just individual words.
                  - Your grade should be either 'yes' or 'no' to indicate whether the document is relevant to the question or not.
             """
grade_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYS_PROMPT),
        ("human", """Retrieved document:
                     {document}

                     User question:
                     {question}
                  """),
    ]
)

# Build grader chain
doc_grader = (grade_prompt
                  |
              structured_llm_grader)

In [ ]:
query = "What is PEFT?"
topk_docs = similarity_threshold_retriever.invoke(query)
for doc in topk_docs:
    print(doc.page_content[:200])
    print('GRADE:', doc_grader.invoke({"question": query, "document": doc.page_content}))
    print()

In [ ]:
query = "what is chain of thought prompting?"
topk_docs = similarity_threshold_retriever.invoke(query)
for doc in topk_docs:
    print(doc.page_content[:200])
    print('GRADE:', doc_grader.invoke({"question": query, "document": doc.page_content}))
    print()

In [ ]:
query = "what are popular patterns for Agentic AI"
topk_docs = similarity_threshold_retriever.invoke(query)
for doc in topk_docs:
    print(doc.page_content[:200])
    print('GRADE:', doc_grader.invoke({"question": query, "document": doc.page_content}))
    print()

In [ ]:
query = "Explain self-attention in detail"
topk_docs = similarity_threshold_retriever.invoke(query)
for doc in topk_docs:
    print(doc.page_content[:200])
    print('GRADE:', doc_grader.invoke({"question": query, "document": doc.page_content}))
    print()

### Create a QA RAG Workflow

This can take in a user query, list of retrived context documents and use a standard RAG workflow where an LLM uses these context documents to generate a contextual response for the user query

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from operator import itemgetter

# Create RAG prompt for response generation
prompt = """You are an assistant for question-answering tasks on popular research topics.
            Use the following pieces of retrieved context to answer the question.
            If no context is present or if you don't know the answer, just say that you don't know the answer.
            Do not make up the answer unless it is there in the provided context.
            Give a detailed answer and to the point answer with regard to the question.

            Question:
            {question}

            Context:
            {context}

            Answer:
         """
prompt_template = ChatPromptTemplate.from_template(prompt)

# Initialize connection with GPT-4o
chatgpt = ChatOpenAI(model_name='gpt-4o', temperature=0)
# Used for separating context docs with new lines
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# create QA RAG chain
qa_rag_chain = (
    {
        "context": (itemgetter('context')
                        |
                    RunnableLambda(format_docs)),
        "question": itemgetter('question')
    }
      |
    prompt_template
      |
    chatgpt
      |
    StrOutputParser()
)

In [ ]:
query = "what is chain of thought prompting?"
topk_docs = similarity_threshold_retriever.invoke(query)
result = qa_rag_chain.invoke(
    {"context": topk_docs, "question": query}
)
print(result)

In [ ]:
query = "Explain self-attention in detail"
topk_docs = similarity_threshold_retriever.invoke(query)
result = qa_rag_chain.invoke(
    {"context": topk_docs, "question": query}
)
print(result)

In [ ]:
query = "What are the most popular design patterns for Agentic AI?"
topk_docs = similarity_threshold_retriever.invoke(query)
result = qa_rag_chain.invoke(
    {"context": topk_docs, "question": query}
)
print(result)

In [ ]:
query = "what is time-series forecasting?"
topk_docs = similarity_threshold_retriever.invoke(query)
result = qa_rag_chain.invoke(
    {"context": topk_docs, "question": query}
)
print(result)

### Create a Query Rephraser Workflow

This uses an LLM to rephrase the given user query (if needed) and make it more optimized for web search

In [ ]:
# LLM for question rewriting
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# Prompt template for rewriting
SYS_PROMPT = """Act as a question re-writer and perform the following task:
                 - Convert the following input question to a better version that is optimized for web search.
                 - Before re-writing, look at the input question and try to reason about the underlying semantic intent / meaning and then re-write it.
             """
re_write_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYS_PROMPT),
        ("human", """Here is the initial question:
                     {question}

                     Formulate an improved question.
                  """,
        ),
    ]
)
# Create rephraser chain
question_rewriter = (re_write_prompt
                        |
                       llm
                        |
                     StrOutputParser())

In [ ]:
query = "what are popular patterns for Agentic AI?"
question_rewriter.invoke({"question": query})

### Create a Web Search Tool

Here we will be using the [Tavily API](https://tavily.com/#api) for our web searches to search and extract content from web pages in the top search results.

In [ ]:
from langchain_tavily._utilities import TavilySearchAPIWrapper
from langchain_core.tools import tool

tavily_search = TavilySearchAPIWrapper()

@tool
def search_web(query: str) -> list:
    """Search the web for a query. Userful for general information or general news"""
    results = tavily_search.raw_results(query=query,
                                        max_results=6,
                                        search_depth='advanced',
                                        include_answer=False,
                                        include_raw_content=True)
    results = [r['raw_content'] for r in results['results']]
    results = [doc for doc in results if doc is not None] # remove blank page content
    return results

## Define the Agent State Schema

Here we define the key state schema that maintains the agent's state across different steps of execution in the LangGraph workflow.

We define a `GraphState` typed dictionary to track relevant information during the execution of the Agentic Corrective RAG System:

- **question**: The query asked by the user.
- **generation**: The final response generated by the LLM, based on retrieved context documents and / or web search.
- **web_search_needed**: A flag (`yes` or `no`) indicating whether a web search fallback is required due to insufficient or irrelevant context from the vector database.
- **documents**: A list of context documents used to answer the query. These may be retrieved from either the internal vector database and (or) a web search tool depending on the workflow path.


In [ ]:
from typing import List
from typing_extensions import TypedDict

class GraphState(TypedDict):
    """
    Represents the state of our graph.

    Attributes:
        question: question
        generation: LLM response generation
        web_search_needed: flag of whether to add web search - yes or no
        documents: list of context documents
    """

    question: str
    generation: str
    web_search_needed: str
    documents: List[str]

## Plan the Agent Workflow Structure

This is the Agent workflow we will be using

![](https://i.imgur.com/xKC0o1A.png)

Next up we will define python functions for each of the nodes in this Agent graph

## Create Node Functions

Each function below represents a stage in processing a user query in the Agentic Corrective RAG System:

1. **retrieve**: Retrieves a set of potentially relevant documents from a vector database based on the user's original query.

2. **grade_documents**: Uses an LLM-based grading to evaluate whether each retrieved document is relevant to the query. Also sets the `web_search_needed` flag variable.
  - Computes the %age of relevant context documents.
  - If this is > a threshold (0.5 is our case) then `web_search_needed=No`
  - However if this %age of relevant documents is <= the threshold (0.5 in our case) or no documents were retrieved for the query then `web_search_needed=Yes`

3. **rewrite_query**: This function rephrases the original query into a version that is better optimized for web search. This ensures better recall during the fallback web search step.

4. **web_search**: Performs a web search using the rephrased query and collects external context documents for RAG response generation.

5. **generate_answer**: Takes the query and the most relevant set of context documents — whether from the vector DB and / or the web — and generates a grounded, final response using a constrained RAG prompt.


### Create the retrieve function

Retrieves a set of potentially relevant documents from a vector database based on the user's original query.

In [ ]:
def retrieve(state):
    """
    Retrieve documents

    Args:
        state (dict): The current graph state

    Returns:
        state (dict): New key added to state, documents - that contains retrieved context documents
    """
    print("---RETRIEVAL FROM VECTOR DB---")
    question = state["question"]

    # Retrieval
    documents = similarity_threshold_retriever.invoke(question)
    return {"documents": documents}

### Create the grade_documents function

Uses an LLM-based grading to evaluate whether each retrieved document is relevant to the query. Also sets the `web_search_needed` flag variable.
  - Computes the %age of relevant context documents.
  - If this is > a threshold (0.5 is our case) then `web_search_needed=No`
  - However if this %age of relevant documents is <= the threshold (0.5 in our case) or no documents were retrieved for the query then `web_search_needed=Yes`

In [ ]:
def grade_documents(state):
    """
    Determines whether the retrieved documents are relevant to the question
    by using an LLM Grader.

    If <= 50% documents are relevant to question or documents are empty - Web Search needs to be done
    If > 50% documents are relevant to question - Web Search is not needed
    Helps filtering out irrelevant documents

    Args:
        state (dict): The current graph state

    Returns:
        state (dict): Updates documents key with only filtered relevant documents
    """

    print("---CHECK DOCUMENT RELEVANCE TO QUESTION---")
    question = state["question"]
    documents = state["documents"]
    RELEVANCE_THRESHOLD = 0.5 # what %age of retrieved documents should at least be relevant

    # Score each doc
    filtered_docs = []
    web_search_needed = "No"
    total_irrelevant = 0
    if documents:
        for d in documents:
            score = doc_grader.invoke(
                {"question": question, "document": d.page_content}
            )
            grade = score.binary_score
            if grade == "yes":
                print("---GRADE: DOCUMENT RELEVANT---")
                filtered_docs.append(d) # store only relevant documents
            else:
                print("---GRADE: DOCUMENT NOT RELEVANT---")
                total_irrelevant += 1 # count number of irrelevant documents

        relevance_frac = 1 - (total_irrelevant / len(documents)) # compute %age of relevant documents
        if relevance_frac <= RELEVANCE_THRESHOLD:
            print("---SEVERAL DOCUMENTS ("+str((1-relevance_frac)*100)+"%) ARE NOT RELEVANT TO QUESTION - WEB SEARCH NEEDED---")
            web_search_needed = "Yes"
        else:
            print("---MOST DOCUMENTS ("+str(relevance_frac*100)+"%) ARE RELEVANT TO QUESTION - WEB SEARCH NOT NEEDED---")

    else:
        print("---NO DOCUMENTS RETRIEVED - WEB SEARCH NEEDED---")
        web_search_needed = "Yes"

    return {"documents": filtered_docs, "web_search_needed": web_search_needed}

### Create the rewrite_query function

This function rephrases the original query into a version that is better optimized for web search. This ensures better recall during the fallback web search step.

In [ ]:
def rewrite_query(state):
    """
    Rewrite the query to produce a better question.

    Args:
        state (dict): The current graph state

    Returns:
        state (dict): Updates question key with a re-phrased or re-written question
    """

    print("---REWRITE QUERY---")
    question = state["question"]

    # Re-write question
    better_question = question_rewriter.invoke({"question": question})
    return {"question": better_question}

### Create the web_search function

Performs a web search using the rephrased query and collects external context documents for RAG response generation.

In [ ]:
from langchain.schema import Document

def web_search(state):
    """
    Web search based on the re-written question.

    Args:
        state (dict): The current graph state

    Returns:
        state (dict): Updates documents key with appended web results
    """

    print("---WEB SEARCH---")
    question = state["question"]
    documents = state["documents"]

    # Web search and extract content from top search results
    docs = search_web.invoke(question)
    web_results = [Document(page_content=d) for d in docs]
    documents.extend(web_results)

    return {"documents": documents}

### Create the generate_answer function

Takes the query and the most relevant set of context documents — whether from the vector DB and / or the web — and generates a grounded, final response using a constrained RAG prompt.

In [ ]:
def generate_answer(state):
    """
    Generate answer from context document using LLM

    Args:
        state (dict): The current graph state

    Returns:
        state (dict): New key added to state, generation, that contains LLM generation
    """
    print("---GENERATE ANSWER---")
    question = state["question"]
    documents = state["documents"]

    # RAG generation
    generation = qa_rag_chain.invoke({"context": documents, "question": question})
    return {"generation": generation}

## Build and Compile the Agent Graph Workflow

We construct a LangGraph agent workflow with the following sequence of nodes:

1. **retrieve** → Retrieves documents from a vector database based on the user’s original query.
2. **grade_documents** → Uses an LLM grader to assess whether the retrieved documents are relevant to the query.
3. If **> 50% documents are relevant**, proceed directly to **generate_answer**.
4. If **<= 50% documents are relevant**, route the flow through a corrective fallback:
   - **rewrite_query** → Rephrases the original query to improve information retrieval from the web.
   - **web_search** → Performs a web search using the rephrased query and collects updated context.
5. Finally, **generate_answer** → Uses the best available context (from DB or web) to produce a grounded and accurate response.


In [ ]:
from langgraph.graph import END, StateGraph

# Create a typed LangGraph state graph using the custom GraphState
agentic_rag = StateGraph(GraphState)

# Register each functional node in the graph that represents a step in the agent workflow
agentic_rag.add_node("retrieve", retrieve)  # retrieve
agentic_rag.add_node("grade_documents", grade_documents)  # grade documents
agentic_rag.add_node("rewrite_query", rewrite_query)  # transform query
agentic_rag.add_node("web_search", web_search)  # web search
agentic_rag.add_node("generate_answer", generate_answer)  # generate answer

# Define the router function that directs the flow based on web_search_needed flag status
def generate_or_search(state):
    """
    Determines whether to generate an answer, or re-generate a question for web search.

    Args:
        state (dict): The current graph state

    Returns:
        str: Binary decision for next node to call
    """

    print("---ASSESS GRADED DOCUMENTS---")
    web_search_needed = state["web_search_needed"]

    if web_search_needed == "Yes":
        # web search needed as some documents are not relevant
        # We will re-generate a new query
        print("---DECISION: SOME or ALL DOCUMENTS ARE NOT RELEVANT TO QUESTION, REWRITE QUERY---")
        return "rewrite_query"
    else:
        # We have relevant documents, so generate answer
        print("---DECISION: GENERATE RESPONSE---")
        return "generate_answer"


# Define the flow of transitions between the nodes in the graph

# starting point is to retrieve documents from the vector db
agentic_rag.set_entry_point("retrieve")
# grade documents after that
agentic_rag.add_edge("retrieve", "grade_documents")
# either generate RAG response or go towards rewrite query for web search
agentic_rag.add_conditional_edges(
    "grade_documents",
    generate_or_search,
    ["rewrite_query", "generate_answer"]
)
# web search after rewriting query
agentic_rag.add_edge("rewrite_query", "web_search")
# generate answer after web search
agentic_rag.add_edge("web_search", "generate_answer")
# stop agent after generating answer
agentic_rag.add_edge("generate_answer", END)

# Compile
agentic_rag = agentic_rag.compile()

In [ ]:
from IPython.display import Image, display, Markdown

display(Image(agentic_rag.get_graph().draw_mermaid_png()))

## Test the Agentic CRAG System

In [ ]:
query = "what is chain of thought prompting?"
response = agentic_rag.invoke({"question": query})

In [ ]:
display(Markdown(response['generation']))

In [ ]:
response

In [ ]:
query = "What are the most popular design patterns for Agentic AI?"
response = agentic_rag.invoke({"question": query})

In [ ]:
display(Markdown(response['generation']))

In [ ]:
query = "Explain self-attention in detail"
response = agentic_rag.invoke({"question": query})

In [ ]:
display(Markdown(response['generation']))

In [ ]:
query = "what is time series forecasting?"
response = agentic_rag.invoke({"question": query})

In [ ]:
display(Markdown(response['generation']))